# Fourier Transform

This notebook splits a signal into its sinusoidal building blocks. It lifts the "baseband audio" and spectrum work out of the legacy `dsp.ipynb` notebook into a focused module on time-domain intuition, spectral peaks, and synthesis.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## A Voice-Like Signal in Time and Frequency

The Fourier transform answers a simple question: *which sinusoids are present, and how much of each is there?* We start with a synthetic three-tone signal so the time waveform looks busy but the spectrum stays easy to read.

In [ ]:
fs = 44_100
t = np.arange(0, 0.04, 1 / fs)
sig = (
    0.9 * np.cos(2 * np.pi * 220 * t)
    + 0.5 * np.cos(2 * np.pi * 440 * t)
    + 0.2 * np.cos(2 * np.pi * 880 * t + np.pi / 6)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
plot_waveform(sig[:1500], fs=fs, ax=axes[0], title="Time Domain")
plot_spectrum(sig, fs=fs, ax=axes[1], title="Magnitude Spectrum")
axes[1].set_xlim(0, 2000)
axes[1].set_ylim(-100, 5)
plt.tight_layout()

display(Audio(normalize(sig), rate=fs))


## Interactive Spectrum Synthesizer

The legacy notebook already showed that a waveform gets visually complicated long before the spectrum does. This widget makes that connection explicit: drag the component amplitudes and watch narrow spikes rearrange the waveform.

In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def update_synth(f1=220.0, a1=0.9, f2=440.0, a2=0.5, f3=880.0, a3=0.2):
    fs = 44_100
    t = np.arange(0, 0.03, 1 / fs)
    sig = (
        a1 * np.cos(2 * np.pi * f1 * t)
        + a2 * np.cos(2 * np.pi * f2 * t)
        + a3 * np.cos(2 * np.pi * f3 * t)
    )
    axes[0].clear()
    axes[1].clear()
    plot_waveform(sig[:1200], fs=fs, ax=axes[0], title="Waveform")
    plot_spectrum(sig, fs=fs, ax=axes[1], title="Spectrum")
    axes[1].set_xlim(0, 3000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, sig, rate=fs)

controls = widgets.interactive(
    update_synth,
    f1=float_slider(min_value=100, max_value=1200, step=20, value=220, description="f1"),
    a1=float_slider(min_value=0, max_value=1, step=0.05, value=0.9, description="a1"),
    f2=float_slider(min_value=100, max_value=2000, step=20, value=440, description="f2"),
    a2=float_slider(min_value=0, max_value=1, step=0.05, value=0.5, description="a2"),
    f3=float_slider(min_value=100, max_value=3000, step=20, value=880, description="f3"),
    a3=float_slider(min_value=0, max_value=1, step=0.05, value=0.2, description="a3"),
)

display(controls, audio_out)


## Beat Frequencies

Two nearby tones create a slow amplitude wobble called a beat. In the spectrum, nothing mysterious happens: you just see two close lines. In the time domain, their interference makes the envelope pulse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_beats(base_freq=440.0, separation=4.0):
    fs = 44_100
    t = np.arange(0, 1.0, 1 / fs)
    sig = np.cos(2 * np.pi * base_freq * t) + np.cos(2 * np.pi * (base_freq + separation) * t)
    axes[0].clear()
    axes[1].clear()
    plot_waveform(sig[:6000], fs=fs, ax=axes[0], title="Beat Pattern")
    plot_spectrum(sig, fs=fs, ax=axes[1], title="Two Nearby Spectral Lines")
    axes[1].set_xlim(base_freq - 30, base_freq + separation + 30)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, sig, rate=fs)

controls = widgets.interactive(
    update_beats,
    base_freq=float_slider(min_value=200, max_value=1200, step=20, value=440, description="Base Hz"),
    separation=float_slider(min_value=0.5, max_value=20, step=0.5, value=4, description="Delta Hz"),
)
display(controls, audio_out)


## What to Try

- Increase `a3` until the spectrum shows a strong third harmonic and notice how much rougher the waveform sounds.
- Move `f2` close to `f1` and listen for beats before you can clearly hear the two tones separately.
- Set one amplitude to zero and confirm that one spectral spike disappears completely.

## Key Takeaway

The time waveform can look complicated while the spectrum stays simple. That is the core value of the Fourier transform: it gives you a clearer coordinate system for understanding signals.